<a href="https://colab.research.google.com/github/gschoie/ecos-fx-rates/blob/main/%EC%88%98%EC%B6%9C%EC%9E%85(%EC%9E%A0%EC%A0%95)_%EB%B0%A9%EC%82%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install requests pycryptodome pandas openpyxl beautifulsoup4

In [ ]:
import json, math, requests
import pandas as pd
from pathlib import Path
from getpass import getpass
from base64 import b64encode
from urllib.parse import quote
from datetime import datetime
from bs4 import BeautifulSoup
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
from google.colab import files

BASE = "https://www.bandtrass.or.kr"
TARGET_URL = f"{BASE}/customs/total.do?command=CUS001View&viewCode=CUS00401"

USER_ID = "ktbgr11"
PASSWORD = getpass("TRASS 비밀번호 입력: ")

SYSTEM_HS = {
    "K2": ["871000"],
    "K9": ["9301101000", "9301109000"],
    "T-50": ["8802301000", "8802302000", "8802303000", "8802309000",
             "8802401000", "8802402000", "8802403000", "8802409000"],
    "유도탄": ["930621", "930629", "930630", "930690"],
}

SYSTEM_COUNTRIES = {
    "K2": ["전세계(합계)", "폴란드", "사우디아라비아", "튀르키예", "이집트", "이라크", "노르웨이", "호주", "인도네시아", "싱가포르", "미국", "독일", "핀란드", "에스토니아", "루마니아", "아랍에미리트 연합", "라트비아", "네덜란드", "페루", "바레인", "이탈리아", "필리핀", "인도", "스페인", "체코공화국", "말레이시아", "덴마크", "이스라엘", "영국", "태국", "그리스", "파키스탄", "캐나다", "아일랜드", "벨기에", "우루과이", "슬로베니아", "칠레", "오만", "중국", "대만", "프랑스", "스웨덴", "아이슬란드", "멕시코", "일본", "헝가리", "아제르바이잔", "기타"],
    "K9": ["전세계(합계)", "폴란드", "이집트", "핀란드", "에스토니아", "노르웨이", "호주", "인도네시아", "베트남", "체코공화국", "미국", "튀르키예", "이탈리아", "기타"],
    "T-50": ["전세계(합계)", "미국", "호주", "폴란드", "필리핀", "인도네시아", "일본", "몽골", "태국", "아르헨티나", "중국", "영국", "캐나다", "대만", "괌", "독일", "베트남", "프랑스", "이탈리아", "홍콩", "이스라엘", "기타"],
    "유도탄": ["전세계(합계)", "폴란드", "아랍에미리트 연합", "사우디아라비아", "미국", "인도네시아", "튀르키예", "영국", "스웨덴", "오만", "필리핀", "루마니아", "체코공화국", "태국", "캐나다", "카타르", "싱가포르", "이스라엘", "파키스탄", "이라크", "이집트", "말레이시아", "세네갈", "대만", "모로코", "스위스", "멕시코", "호주", "케냐", "독일", "오스트리아", "뉴질랜드", "피지", "브라질", "페루", "핀란드", "코트디봐르", "프랑스", "콜롬비아", "이탈리아", "스페인", "몽골", "부르키나 파소", "가봉", "가이아나", "니제르", "에스토니아", "요르단", "남아프리카공화국", "노르웨이", "라트비아", "토고", "홍콩", "카자흐스탄", "과테말라", "베트남", "몰타", "파나마", "방글라데시", "모잠비크", "인도", "일본", "네덜란드", "우루과이", "적도 기니", "러시아 연방", "아르헨티나", "슬로베니아", "스리랑카", "쿠웨이트", "그리스", "챠드", "트리니다드 토바고", "아이슬란드", "불가리아", "르완다", "리비아", "우즈베키스탄", "아제르바이잔", "중국", "벨기에", "덴마크", "슬로바키아", "안도라", "기타"],
}

AES_KEY = b"bandtrassorkr999"
AES_IV = b"bandtrassorkr999"

def encrypt_pw(password):
    cipher = AES.new(AES_KEY, AES.MODE_CBC, AES_IV)
    return b64encode(cipher.encrypt(pad(password.encode(), AES.block_size))).decode()

def chunks(xs, n=3):
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

def excel_serial(dt):
    return (dt - datetime(1899, 12, 30)).days

pw_enc = encrypt_pw(PASSWORD)

s = requests.Session()
s.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
})

rt_encoded = quote(TARGET_URL, safe="")
login_page_url = f"{BASE}/login.do?rtUrl={quote(rt_encoded, safe='')}"

print("1. 로그인")
s.get(login_page_url).raise_for_status()

r = s.post(
    f"{BASE}/login.do?command=loginPerson",
    data={"ID": USER_ID, "PW": pw_enc},
    headers={"X-Requested-With": "XMLHttpRequest", "Referer": login_page_url},
)
r.raise_for_status()
print(r.json())

final_login_url = (
    f"{BASE}/login.do?command=login&returnPage=&rtCode="
    f"&rtUrl={rt_encoded}&ID={quote(USER_ID)}&PW={quote(pw_enc, safe='')}"
)
r = s.get(final_login_url, allow_redirects=True)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")
def hidden_value(name, default=""):
    tag = soup.select_one(f"#{name}")
    return tag.get("value", default) if tag else default

undecide_max_date = hidden_value("UndecideMaxDate")
undecide_batch_dtm = hidden_value("UndecideBatchDtm", undecide_max_date)
year = int(hidden_value("MSF_MAX_YEAR", undecide_max_date[:4] or "2026"))
month = int(undecide_max_date[4:6]) if undecide_max_date else 5

month_date = datetime(year, month, 1)
quarter = math.ceil(month / 3)
quarter_label = f"{str(year)[2:]}.{quarter}Q"
period_label = f"{year}.{month:02d}"

ajax_headers = {
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "X-Requested-With": "XMLHttpRequest",
    "Origin": BASE,
    "Referer": TARGET_URL,
}

def query_codes(codes, export_import):
    frames = []
    for batch in chunks(codes, 3):
        payload = {
            "command": "CUS00401Detail",
            "search": "Y",
            "COL_NAME": "",
            "EXCEL_LOG": "",
            "MENU_CODE": "CUS00401",
            "MSF_MAX_YEAR": str(year),
            "UndecideMaxDate": undecide_max_date,
            "UndecideBatchDtm": undecide_batch_dtm,
            "page": "1",
            "token_excel": "",
            "SEARCH_YEAR": str(year),
            "grid_type": "A",
            "S0010001": export_import,  # E 수출, I 수입
            "UNIT": "1",
            "SelectCd": ",".join(batch),
            "hscd_unit": str(len(batch[0])),
            "NatnCd": "",
            "pageRowCount": "99999",
        }
        r = s.post(f"{BASE}/customs/total.do", data=payload, headers=ajax_headers)
        r.raise_for_status()
        data = r.json()["gridData"]
        data = json.loads(data) if isinstance(data, str) else data
        if data:
            frames.append(pd.DataFrame(data))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print("2. 무기체계별 잠정치 조회")
system_data = {}
for system, codes in SYSTEM_HS.items():
    exp = query_codes(codes, "E")
    imp = query_codes(codes, "I")
    system_data[system] = {"E": exp, "I": imp}
    print(system, "export", len(exp), "import", len(imp))

def num(s):
    return pd.to_numeric(s, errors="coerce").fillna(0)

wb = Workbook()
ws = wb.active
ws.title = "원천데이터(로우)"

thin = Side(style="thin", color="D9D9D9")
border = Border(left=thin, right=thin, top=thin, bottom=thin)
header_fill = PatternFill("solid", fgColor="D9EAF7")
group_fill = PatternFill("solid", fgColor="E2F0D9")
center = Alignment(horizontal="center", vertical="center")
right = Alignment(horizontal="right", vertical="center")

raw_headers = ["대분류", "소분류", "연", "YY.MM", "YY.QQ", "기간", "수출금액", "수출중량",
               "수입금액(달러)", "수입중량(kg)", "무역수지(달러)", "국가명", "품목코드", "품목명"]
ws.append(raw_headers)

all_raw = []

for system, data in system_data.items():
    exp = data["E"].copy()
    imp = data["I"].copy()

    rows = {}

    for _, r0 in exp.iterrows():
        key = (str(r0["NATN_NAME"]), str(r0["HS_CD"]), str(r0.get("HS_NAME", "-")))
        rows.setdefault(key, {"export_amt": 0, "export_wgt": 0, "import_amt": 0, "import_wgt": 0})
        rows[key]["export_amt"] += float(r0["AMT_OF_DLR"])
        rows[key]["export_wgt"] += float(r0["NET_WGHT"])

    for _, r0 in imp.iterrows():
        key = (str(r0["NATN_NAME"]), str(r0["HS_CD"]), str(r0.get("HS_NAME", "-")))
        rows.setdefault(key, {"export_amt": 0, "export_wgt": 0, "import_amt": 0, "import_wgt": 0})
        rows[key]["import_amt"] += float(r0["AMT_OF_DLR"])
        rows[key]["import_wgt"] += float(r0["NET_WGHT"])

    total_export_amt = sum(v["export_amt"] for v in rows.values())
    total_export_wgt = sum(v["export_wgt"] for v in rows.values())
    total_import_amt = sum(v["import_amt"] for v in rows.values())
    total_import_wgt = sum(v["import_wgt"] for v in rows.values())

    all_raw.append([system, "전세계(합계)", "-", "소분류 통합",
                    total_export_amt, total_export_wgt, total_import_amt, total_import_wgt])

    for (country, code, name), v in rows.items():
        all_raw.append([system, country, code, name,
                        v["export_amt"], v["export_wgt"], v["import_amt"], v["import_wgt"]])

for system, country, code, name, ex_amt, ex_wgt, im_amt, im_wgt in all_raw:
    ws.append([
        "방위산업", system, f"{year}년", month_date, quarter_label, period_label,
        ex_amt, ex_wgt, im_amt, im_wgt, ex_amt - im_amt, country, code, name
    ])

for row in ws.iter_rows():
    for c in row:
        c.border = border
        c.alignment = center if c.column <= 6 or c.column >= 12 else right
        if c.row == 1:
            c.fill = header_fill
            c.font = Font(bold=True)

for r in range(2, ws.max_row + 1):
    ws.cell(r, 4).number_format = "yy.mm"
    for c in range(7, 12):
        ws.cell(r, c).number_format = '#,##0'

for c in range(1, 15):
    ws.column_dimensions[get_column_letter(c)].width = 16
ws.column_dimensions["N"].width = 28

def build_country_sums(system):
    countries = SYSTEM_COUNTRIES[system]
    sums = {c: {"amount_mil": 0.0, "weight_ton": 0.0} for c in countries}

    for row in all_raw:
        sys, country, code, name, ex_amt, ex_wgt, im_amt, im_wgt = row
        if sys != system:
            continue

        bucket = country if country in sums else "기타"
        sums[bucket]["amount_mil"] += ex_amt / 1_000_000
        sums[bucket]["weight_ton"] += ex_wgt / 1_000

    return sums

def make_pivot_sheet(name, system, period_value, period_fmt=None):
    ws = wb.create_sheet(name)
    countries = SYSTEM_COUNTRIES[system]
    n = len(countries)

    starts = [3, 3+n, 3+n*2, 3+n*3]
    titles = ["수출금액(백만달러)", "수출중량(톤)", "ASP(백만달러/톤)", "수출금액의 YoY"]

    for start, title in zip(starts, titles):
        end = start + n - 1
        ws.merge_cells(start_row=1, start_column=start, end_row=1, end_column=end)
        cell = ws.cell(1, start, title)
        cell.fill = group_fill
        cell.font = Font(bold=True)
        cell.alignment = center

        for i, country in enumerate(countries):
            h = ws.cell(2, start+i, country)
            h.fill = header_fill
            h.font = Font(bold=True)
            h.alignment = center
            h.border = border

    ws["A2"] = "연"
    ws["B2"] = "YY.MM" if "분기" not in name else "YY.QQ"
    ws["A3"] = "전체 누적"
    ws["B3"] = "합계"
    ws["A4"] = f"{year}년"
    ws["B4"] = period_value

    if period_fmt:
        ws["B4"].number_format = period_fmt

    sums = build_country_sums(system)

    for row_idx in [3, 4]:
        for i, country in enumerate(countries):
            amount = sums[country]["amount_mil"]
            weight = sums[country]["weight_ton"]
            asp = (amount / weight) if weight else 0

            ws.cell(row_idx, starts[0]+i, amount)
            ws.cell(row_idx, starts[1]+i, weight)
            ws.cell(row_idx, starts[2]+i, asp)
            ws.cell(row_idx, starts[3]+i, 0)

    for row in ws.iter_rows(min_row=1, max_row=4, max_col=starts[-1]+n-1):
        for c in row:
            c.border = border
            c.alignment = center if c.column <= 2 else right
            if c.row in [1, 2]:
                c.font = Font(bold=True)
            if c.row >= 3 and c.column >= 3:
                c.number_format = '#,##0.00'

    ws.column_dimensions["A"].width = 12
    ws.column_dimensions["B"].width = 12
    for col in range(3, starts[-1]+n):
        ws.column_dimensions[get_column_letter(col)].width = 14

for system in ["K2", "K9", "T-50", "유도탄"]:
    make_pivot_sheet(f"피벗_{system}", system, month_date, "yy.mm")
    make_pivot_sheet(f"피벗_분기_{system}", system, quarter_label)

out = f"수출입데이터_방산_잠정_{year}{month:02d}.xlsx"
wb.save(out)

print("완료:", out)
files.download(out)